# **Customer Personality Analysis With Customer Lifetime Value**

## **Introduction**



### **GOAL**

Compute **Customer Lifetime Value (CLV)** refers to the total expected revenue a business can earn from a customer throughout their relationship. The longer a customer remains active and the more frequently they purchase, the higher their CLV.

A common way to estimate customer value over time is the **RFM model**, which considers:

- __Recency (R)__ : How recently the customer made a purchase.

- __Frequency (F)__ : How often the customer makes purchases.

- __Monetary (M)__ : How much the customer spends.

- __T (Tenure)__ : The duration between their first purchase and the end of the study period.


#### **Customer Personality Dataset Description:**

The Customer Personality Analysis dataset is a Kaggle dataset designed for customer segmentation, containing demographic details, spending habits, and product preferences of retail customers. It's widely used for clustering, profiling, predict campaign success, and marketing strategy optimization.


### **Feature Distribution:**

**1. Customer Information**
- ID – Unique identifier for each customer
- Year_Birth – Customer's birth year
- Education – Level of education
- Marital_Status – Marital status

**2. Financial Details**
- Income – Annual household income
- Kidhome – Number of children at home
- Teenhome – Number of teenagers at home

**3. Purchase Behavior**
- Recency – Days since last purchase
- MntWines – Amount spent on wine
- MntFruits – Amount spent on fruits
- MntMeatProducts – Amount spent on meat products
- MntFishProducts – Amount spent on fish products
- MntSweetProducts – Amount spent on sweets
- MntGoldProds – Amount spent on gold products

**4. Purchase Channels**
- NumDealsPurchases – Number of purchases made with discount
- NumWebPurchases – Purchases made online
- NumCatalogPurchases – Purchases via catalog
- NumStorePurchases – Purchases in-store
- NumWebVisitsMonth – Monthly website visits

**5. Campaign Response**
- AcceptedCmp1–AcceptedCmp5 – Acceptance of previous campaigns
- Response – Response to the last campaign (Target variable)

**6. Customer Engagement**
- Complain – Whether the customer complained
- Dt_Customer – Date of customer enrollment


**This dataset represents the customers profile and behaviour across multiple years, where each row corresponds to one customer. It includes demographic details, family composition, income, purchasing behaviour and product spending.**

### **1. Data Ingestion & Understanding**

In [35]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display
from IPython.display import Markdown


In [36]:
# display setting
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

import warnings
warnings.simplefilter("ignore", category=FutureWarning)

# Display markdown formatted output like bold, italic bold etc.'''
def display_md(string):
    display(Markdown(string))

    

In [37]:
# Define paths to read files
parent_path = Path.cwd().parent
data_path = parent_path.joinpath('data', 'raw')

files = []
for file in data_path.rglob('*.csv'):
    files.append(file)
    print(files.index(file), ' ', file.name)

0   marketing_campaign.csv


In [76]:
display_md("**Loading Raw Dataset...**")
try:
    customer_data = pd.read_csv(files[0], sep='\t')
    
    # standardize the column names
    customer_data.columns = customer_data.columns.str.strip().str.replace(" ", "_").str.replace(".", "_")

    # Conversion of 'Dt_Customer' into datetime
    customer_data['Dt_Customer'] = pd.to_datetime(customer_data['Dt_Customer'], format='%d-%m-%Y')
    display_md(f"**Dataset Shape : {customer_data.shape}**")
    display_md(f"**Number of duplicate rows in dataset : {customer_data.duplicated().sum()}**")
    
except Exception as e:
    print(f"Error raised while loading file : {e}")

**Loading Raw Dataset...**

**Dataset Shape : (2240, 29)**

**Number of duplicate rows in dataset : 0**

In [78]:
customer_data.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


In [79]:
display_md(f"**Customer Information :**")
customer_data.info()

**Customer Information :**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID                   2240 non-null   int64         
 1   Year_Birth           2240 non-null   int64         
 2   Education            2240 non-null   object        
 3   Marital_Status       2240 non-null   object        
 4   Income               2216 non-null   float64       
 5   Kidhome              2240 non-null   int64         
 6   Teenhome             2240 non-null   int64         
 7   Dt_Customer          2240 non-null   datetime64[ns]
 8   Recency              2240 non-null   int64         
 9   MntWines             2240 non-null   int64         
 10  MntFruits            2240 non-null   int64         
 11  MntMeatProducts      2240 non-null   int64         
 12  MntFishProducts      2240 non-null   int64         
 13  MntSweetProducts     2240 non-nul

In [80]:
## Column Formatting & Consistency
dtype_df = pd.DataFrame(
    index=customer_data.columns, columns=['dtype', 'nunique', 'unique']
)
dtype_df['dtype'] = customer_data.dtypes
dtype_df['nunique'] = customer_data.nunique()
dtype_df['unique'] = [customer_data[col].unique() for col in dtype_df.index]

display(dtype_df)

,dtype,nunique,unique
ID,int64,2240,"[5524, 2174, 4141, 6182, 5324, 7446, 965, 6177..."
Year_Birth,int64,59,"[1957, 1954, 1965, 1984, 1981, 1967, 1971, 198..."
Education,object,5,"[Graduation, PhD, Master, Basic, 2n Cycle]"
Marital_Status,object,8,"[Single, Together, Married, Divorced, Widow, A..."
Income,float64,1974,"[58138.0, 46344.0, 71613.0, 26646.0, 58293.0, ..."
Kidhome,int64,3,"[0, 1, 2]"
Teenhome,int64,3,"[0, 1, 2]"
Dt_Customer,datetime64[ns],663,"[2012-09-04 00:00:00, 2014-03-08 00:00:00, 201..."
Recency,int64,100,"[58, 38, 26, 94, 16, 34, 32, 19, 68, 11, 59, 8..."
MntWines,int64,776,"[635, 11, 426, 173, 520, 235, 76, 14, 28, 5, 6..."
